# 00 - CPV Sector Reference (build from official source)

**Purpose.** The `sector_code` feature is the **CPV 2008 division** - the first two digits of a contract's Common Procurement Vocabulary code. In the raw data these appear as bare numbers (`45`, `33`, …). This notebook attaches human-readable names to them.

**Why a separate notebook.** Labels are needed in several notebooks (01 EDA, 04 logistic-regression coefficients, 05/06 feature importance). Rather than hard-code a label dictionary by hand, this notebook **derives the labels directly from the official EU CPV 2008 code list** and writes them to `data/reference/cpv_sectors.csv`. The later notebooks just load that CSV.

**Provenance.** Every label is read verbatim from the official file - nothing is typed by hand, shortened, or guessed. The English and Bulgarian titles are exactly as published.

**Source file.** `data/reference/cpv_2008_ver_2013.xlsx` - Common Procurement Vocabulary (CPV) 2008, official multilingual code list, published by the European Commission via TED/SIMAP (https://ted.europa.eu/en/simap/cpv). Legal basis: Regulation (EC) No 2195/2002, as amended by Regulation (EC) No 213/2008.

In [1]:
import pandas as pd

REFERENCE_DIR = "../data/reference"
CPV_XLSX = f"{REFERENCE_DIR}/cpv_2008_ver_2013.xlsx"
OUTPUT_CSV = f"{REFERENCE_DIR}/cpv_sectors.csv"

## 1. Read the official code list and isolate the divisions

The sheet `CPV codes` holds one row per CPV code across all EU languages. A code looks like `45000000-7` (eight digits plus a check digit). A **division** is a top-level code whose eight-digit part ends in `000000`. We match those rows and take the first two digits as `sector_code`.

In [2]:
import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    cpv = pd.read_excel(CPV_XLSX, sheet_name="CPV codes", dtype=str)

cpv.columns = [c.strip() for c in cpv.columns]
cpv["code8"] = cpv["CODE"].str.split("-").str[0]
divisions = cpv[cpv["code8"].str.match(r"^\d{2}000000$")].copy()
divisions["sector_code"] = divisions["code8"].str[:2].astype(int)
divisions = divisions[["sector_code", "CODE", "EN", "BG"]].rename(
    columns={"CODE": "cpv_code", "EN": "label_en", "BG": "label_bg"}
)
divisions["label_en"] = divisions["label_en"].str.strip()
divisions["label_bg"] = divisions["label_bg"].str.strip()
divisions = divisions.sort_values("sector_code").reset_index(drop=True)
print(f"{len(divisions)} divisions read from the official file.")
divisions

45 divisions read from the official file.


,sector_code,cpv_code,label_en,label_bg
0,3,03000000-1,"Agricultural, farming, fishing, forestry and r...","Продукти на земеделието, животновъдството, риб..."
1,9,09000000-3,"Petroleum products, fuel, electricity and othe...","Нефтопродукти, горива, електричество и други е..."
2,14,14000000-1,"Mining, basic metals and related products","Продукти на минната индустрия, основни метали ..."
3,15,15000000-8,"Food, beverages, tobacco and related products","Хранителни продукти, напитки, тютюн и свързани..."
4,16,16000000-5,Agricultural machinery,Селскостопански машини
5,18,18000000-9,"Clothing, footwear, luggage articles and acces...","Облекло, обувни изделия, пътни артикули и аксе..."
6,19,19000000-6,"Leather and textile fabrics, plastic and rubbe...","Кожени и текстилни изделия, пластмасови и кауч..."
7,22,22000000-0,Printed matter and related products,Печатни материали и свързани с тях продукти
8,24,24000000-4,Chemical products,Химически продукти
9,30,30000000-9,"Office and computing machinery, equipment and ...","Компютърни и офис машини, оборудване и принадл..."


## 2. Write the shared reference CSV

Later notebooks (01, 04, 05, 06) load this CSV instead of re-parsing the Excel. Committing the CSV also makes the label set diff-able in git. Both the English and Bulgarian official titles are kept verbatim - no shortening or editing is applied.

In [3]:
divisions.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"Wrote {len(divisions)} rows to {OUTPUT_CSV}")
print("Columns:", list(divisions.columns))

Wrote 45 rows to ../data/reference/cpv_sectors.csv
Columns: ['sector_code', 'cpv_code', 'label_en', 'label_bg']


## 3. How the other notebooks use this

Load the CSV and build lookup dicts:

```python
import pandas as pd
ref = pd.read_csv("../data/reference/cpv_sectors.csv")
labels_en = dict(zip(ref.sector_code, ref.label_en))

def sector_display(code):              # "Construction work (45)"
    return f"{labels_en[int(code)]} ({int(code)})"
```

**Integrity check to run wherever the labels are applied** - fail loudly if a code in the data has no official label, rather than silently showing "Unknown":

```python
data_codes = set(df["sector_code"].astype(int).unique())
missing = sorted(data_codes - set(labels_en))
assert not missing, f"Codes in data absent from official CPV file: {missing}"
```

**Findings & decisions.** `sector_code` is the CPV *division* (two-digit rollup) - a deliberately coarse category chosen to keep cardinality manageable; finer CPV levels (group/class) would explode the feature space. Labels are sourced verbatim from the official EU CPV 2008 list, with no editing applied. Some titles are long; for crowded chart axes, prefer horizontal bars or rotate/wrap tick labels rather than abbreviating the official name.